# TP4 : Prédiction de notes d'hôtels avec ModernBERT

## Objectif
Prédire la note (1 à 5 étoiles) d'un avis d'hôtel à partir du texte de la review, en utilisant un modèle de type Transformer (ModernBERT).

## Dataset
| Ensemble | Taille |
|----------|--------|
| Train    | 18 491 |
| Validation | 1 000 |
| Test     | 1 000  |

Le dataset est **déséquilibré** : les notes 4★ et 5★ représentent ~73% des données d'entraînement.

| Note | Nombre | Proportion |
|------|--------|------------|
| 1★   | 1 310  | 7.1%       |
| 2★   | 1 630  | 8.8%       |
| 3★   | 1 982  | 10.7%      |
| 4★   | 5 465  | 29.6%      |
| 5★   | 8 104  | 43.8%      |

## Approche retenue : CORN (Conditional Ordinal Regression)

### Pourquoi CORN plutôt qu'une classification classique ?

Les notes d'hôtel sont **ordinales** : 1 < 2 < 3 < 4 < 5. Une classification standard (CrossEntropy) traite les classes comme indépendantes — se tromper de 1★ vs 5★ coûte autant que 4★ vs 5★.

**CORN** (Conditional Ordinal Regression Network) résout ce problème :
- Au lieu de prédire directement une classe parmi 5, le modèle répond à **4 questions binaires** :
  - "La note est-elle > 1 ?" → oui/non
  - "La note est-elle > 2 ?" → oui/non
  - "La note est-elle > 3 ?" → oui/non
  - "La note est-elle > 4 ?" → oui/non
- La note finale = nombre de "oui" + 1 (car on commence avec 0 donc nos note son de 0 à 4)
- Exemple : [oui, oui, oui, non] → note = 4★

Cela force le modèle à respecter l'ordre naturel des notes.

### Gestion du déséquilibre : Class Weights

Comme les notes 4★ et 5★ dominent le dataset, le modèle pourrait apprendre à toujours prédire 5★ (ce qui donnerait déjà ~44% d'accuracy). Pour contrer cela, on utilise des **class weights** (pondération inverse de la fréquence) dans la loss :

| Note | Poids |
|------|-------|
| 1★   | 1.745 |
| 2★   | 1.402 |
| 3★   | 1.153 |
| 4★   | 0.418 |
| 5★   | 0.282 |

Une erreur sur un avis 1★ coûte ~6x plus cher qu'une erreur sur un avis 5★. Cela force le modèle à bien apprendre les classes rares.

**Note :** Nous avons aussi testé l'**oversampling** (sur-représenter les classes rares dans les batchs), mais cela a **dégradé** les résultats (R² passé de 0.73 à 0.69) car la distribution modifiée déstabilisait l'entraînement. Les class weights dans la loss seule sont la meilleure approche.

### Architecture du modèle

```
Texte de la review
       ↓
ModernBERT-base (encodeur Transformer, 768 dims)
       ↓
Mean Pooling (moyenne pondérée des tokens)
       ↓
Linear(768 → 256) + GELU + Dropout(0.3) + LayerNorm
       ↓
Linear(256 → 4)   ← 4 logits CORN
       ↓
Sigmoid > 0.5 → Somme + 1 = Note prédite (1-5)
```

**Pourquoi ModernBERT ?** C'est une version améliorée de BERT, plus rapide et plus performante sur les tâches de NLP. Le **mean pooling** (moyenne de tous les tokens) est recommandé pour ModernBERT, contrairement au token [CLS] utilisé dans BERT classique.

**Pourquoi une tête simplifiée ?** Une seule couche intermédiaire (768→256→4) avec GELU et LayerNorm suffit. Des architectures plus complexes (dual-head, couches multiples) n'ont pas amélioré les résultats dans nos tests.

## Méthodes testées et résultats comparés

Voici le récapitulatif de toutes les approches testées au cours de ce TP :
| Version | Méthode | R² Test | Accuracy | Commentaire |
|---------|---------|---------|----------|-------------|
| Baseline 1 | TF-IDF (ngram=4) + classifieur | 67% | 63% | Approche classique sans deep learning, plafonne vite |
| Baseline 2 | Sentence Embedding + classifieur | 69% | 65% | Meilleur que TF-IDF mais ne capture pas l'ordinalité |
| v1 | ModernBERT + CrossEntropy classique | ~0.65 | ~62% | Fine-tuning deep learning, mais CrossEntropy ignore l'ordre des notes |
| v2 | ModernBERT + CORN + class weights + cosine scheduler + battle royale seeds | **0.7308** | **68.50%** | Meilleur résultat individuel, CORN capture l'ordinalité |
| v3 | v2 + dual-head (CORN + régression) + oversampling agressif + ensemble 3 modèles | 0.7492 | 68.70% | L'ensemble aide (+0.02 R²) mais modèles individuels moins bons |
| v4a | v2 + oversampling sqrt + expected value | 0.6918 | 63.80% | L'oversampling dégrade les résultats |
| **v4b** | **v2 exacte + FP16 + multi-GPU + 10 epochs** | **75%** | **70%** | plus rapide et plus d'epochs |

### Leçons apprises

1. **CORN > CrossEntropy** pour les labels ordinaux (+0.08 R²)
2. **Class weights** dans la loss fonctionnent bien, l'oversampling non
3. **L'ensemble** de modèles apporte ~+0.02 R² mais au prix de 3x le temps d'entraînement et de problèmes de mémoire
4. **Le battle royale de seeds** a permis d'identifier que la seed 2024 donnait les meilleurs résultats
5. **Mean pooling > CLS token** pour ModernBERT
6. **Le cosine scheduler avec warmup** est essentiel pour la convergence

### Optimisations techniques

- **FP16 (Mixed Precision)** : les calculs sont faits en demi-précision (16 bits au lieu de 32), ce qui accélère l'entraînement d'environ 1.5x sans perte de qualité.
- **Multi-GPU (DataParallel)** : le modèle est répliqué sur 2 GPUs, chacune traite la moitié du batch, les gradients sont agrégés → 2x plus rapide.
- **Gradient Checkpointing** : économise la mémoire GPU en recalculant les activations pendant le backward pass au lieu de les stocker.
- **Early Stopping** (patience=3) : on arrête l'entraînement si le R² sur la validation ne s'améliore pas pendant 3 epochs consécutives.

---
# Code complet
---

## 1. Imports et configuration

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os, time, random
from sklearn.metrics import r2_score, mean_absolute_error, confusion_matrix
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

# Seed pour reproductibilité (meilleure seed identifiée par battle royale en v2)
SEED = 2024

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)

# Détection des GPUs
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print(f"GPUs détectés : {N_GPUS}")
for i in range(N_GPUS):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {name} ({mem:.1f} Go)")

# Hyperparamètres (identiques à la v2)
model_id    = "answerdotai/ModernBERT-base"
num_classes = 5
batch_size  = 16
epochs      = 10        # v2 avait 4, on pousse plus loin
patience    = 3         # early stopping
max_length  = 512
dropout     = 0.3

print(f"\nDevice: {device} | batch_size: {batch_size} | epochs: {epochs}")
print(f"Seed: {SEED} | max_length: {max_length} | FP16: activé")

## 2. Chargement des données

In [ ]:
def load_data(file_path):
    with open(os.path.join('data', file_path), 'r') as f:
        return pd.read_csv(f)

data_train = load_data('train_hotel.csv')
data_test  = load_data('test_hotel.csv')
data_val   = load_data('valid_hotel.csv')

print(f"Train: {len(data_train)} | Val: {len(data_val)} | Test: {len(data_test)}")
print(data_train[['Review', 'Rating']].head(3))

# Visualisation du déséquilibre
rating_counts = data_train['Rating'].value_counts().sort_index()
print("\nDistribution des notes (train):")
for r, c in rating_counts.items():
    print(f"  {r}★ : {c:5d} ({100*c/len(data_train):.1f}%)")

plt.figure(figsize=(8, 4))
plt.bar(rating_counts.index, rating_counts.values, color='steelblue')
plt.xlabel('Note'); plt.ylabel('Nombre d\'avis')
plt.title('Distribution des notes dans le dataset d\'entraînement')
plt.xticks([1, 2, 3, 4, 5])
plt.show()

## 3. Tokenisation et DataLoaders

On utilise le tokenizer de ModernBERT pour convertir le texte en tokens. Le `collate_fn` tokenise chaque batch à la volée (plus efficace en mémoire que de tout tokeniser d'un coup).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

class HotelDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    encoded = tokenizer(
        list(texts), padding=True, truncation=True,
        max_length=max_length, return_tensors="pt"
    )
    return encoded, torch.stack(list(labels))

train_labels = torch.tensor(data_train['Rating'].values - 1, dtype=torch.long)
val_labels   = torch.tensor(data_val['Rating'].values - 1, dtype=torch.long)
test_labels  = torch.tensor(data_test['Rating'].values - 1, dtype=torch.long)

# Pas d'oversampling : la distribution naturelle est conservée
train_loader = DataLoader(
    HotelDataset(data_train['Review'].tolist(), train_labels),
    batch_size=batch_size, shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    HotelDataset(data_val['Review'].tolist(), val_labels),
    batch_size=batch_size * 2, shuffle=False,
    collate_fn=collate_fn, num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    HotelDataset(data_test['Review'].tolist(), test_labels),
    batch_size=batch_size * 2, shuffle=False,
    collate_fn=collate_fn, num_workers=0, pin_memory=True
)

## 4. Class Weights

On calcule des poids inversement proportionnels à la fréquence de chaque classe. Cela pénalise davantage les erreurs sur les classes rares (1★, 2★).

In [ ]:
class_counts = np.bincount(train_labels.numpy(), minlength=num_classes).astype(float)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * num_classes  # normalisation
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights:", np.round(class_weights, 3))

## 5. Architecture du modèle

Le modèle combine :
- **ModernBERT-base** comme encodeur de texte (768 dimensions)
- **Mean pooling** : on fait la moyenne pondérée de tous les tokens (meilleur que le token CLS pour ModernBERT)
- **Tête CORN** : une couche cachée (256 neurones, GELU, Dropout, LayerNorm) suivie de 4 neurones de sortie

In [ ]:
class RegressionModel(nn.Module):
    """
    ModernBERT + Mean Pooling + CORN Head.
    Produit 4 logits (K-1 comparaisons binaires) pour la régression ordinale.
    """
    def __init__(self, model_name=model_id, num_classes=5,
                 use_mean_pooling=True, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.use_mean_pooling = use_mean_pooling
        hidden_size = self.bert.config.hidden_size  # 768

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(256),
            nn.Linear(256, num_classes - 1),  # 4 logits CORN
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        if self.use_mean_pooling:
            # Moyenne pondérée par le masque d'attention
            mask   = attention_mask.unsqueeze(-1).expand(out.last_hidden_state.size()).float()
            pooled = torch.sum(out.last_hidden_state * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        else:
            pooled = out.last_hidden_state[:, 0, :]  # CLS token

        return self.head(pooled)

## 6. Fonctions CORN

### Conversion des labels
Un label de note 4 (index 3) est converti en `[1, 1, 1, 0]` :
- "> 1" → oui (1)
- "> 2" → oui (1)  
- "> 3" → oui (1)
- "> 4" → non (0)

### Loss pondérée
On utilise `BCEWithLogitsLoss` (Binary Cross Entropy) sur chacun des 4 logits, pondérée par les class weights. Le **label smoothing** (0.05) adoucit les cibles de 0/1 pour éviter la sur-confiance.

### Prédiction
On applique sigmoid sur les logits, puis on compte combien sont > 0.5 pour obtenir la note.

In [ ]:
def label_to_corn(labels, num_classes):
    """Convertit un label (0-4) en vecteur CORN binaire."""
    return torch.stack([labels > i for i in range(num_classes - 1)], dim=1).float()

def predict_rating_from_logits(logits):
    """Logits CORN → note prédite (1-5)."""
    return torch.sum(torch.sigmoid(logits) > 0.5, dim=1) + 1

def corn_loss_weighted(logits, targets, class_weights, num_classes, device,
                       label_smoothing=0.05):
    """
    BCEWithLogitsLoss pondérée par classe, avec label smoothing.
    """
    corn_targets = label_to_corn(targets, num_classes).to(device)
    
    # Label smoothing : cibles 0/1 → 0.025/0.975
    corn_targets = corn_targets * (1 - label_smoothing) + 0.5 * label_smoothing
    
    # Poids par exemple = poids de sa vraie classe
    sample_weights = class_weights[targets.cpu()].to(device)
    
    bce = nn.BCEWithLogitsLoss(reduction='none')(logits, corn_targets)
    bce_mean = bce.mean(dim=1)  # moyenne sur les 4 logits
    return (bce_mean * sample_weights).mean()  # moyenne pondérée sur le batch

## 7. Construction du modèle + Multi-GPU + FP16

- **DataParallel** : réplique le modèle sur les 2 GPUs automatiquement
- **GradScaler** : gère le FP16 (scaling des gradients pour éviter les underflows)
- **Cosine scheduler avec warmup** : le learning rate monte pendant les 6% premières steps

In [ ]:
set_seed(SEED)
model = RegressionModel(model_id, num_classes=num_classes,
                        use_mean_pooling=True, dropout=dropout).to(device)
model.bert.gradient_checkpointing_enable()

# Multi-GPU si disponible
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel activé sur {N_GPUS} GPUs")

base_model = model.module if N_GPUS > 1 else model

# Optimiseur avec learning rates différenciés
# BERT a un LR plus faible (2e-5) car il est déjà pré-entraîné
# La tête a un LR plus élevé (1e-3) car elle est initialisée aléatoirement
optimizer = torch.optim.AdamW([
    {'params': base_model.bert.parameters(), 'lr': 2e-5},
    {'params': base_model.head.parameters(), 'lr': 1e-3},
], weight_decay=0.01)

total_steps = epochs * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.06 * total_steps),
    num_training_steps=total_steps
)

scaler = GradScaler()  # FP16
print(f"Modèle prêt. Total steps: {total_steps}")

## 8. Entraînement avec Early Stopping

À chaque epoch :
1. **Train** : forward en FP16, backward avec GradScaler, clip des gradients à 1.0
2. **Eval** : calcul du R² et de l'accuracy sur la validation
3. **Early stopping** : si le R² ne s'améliore pas pendant 3 epochs, on arrête et on restaure le meilleur checkpoint

In [ ]:
print("=" * 60)
print(f"ENTRAÎNEMENT — Seed {SEED}")
print("=" * 60)

best_r2       = -float('inf')
best_acc      = 0
best_state    = None
patience_left = patience
history       = {'loss': [], 'r2': [], 'acc': []}

for ep in range(epochs):
    t0 = time.time()

    # ── Train ─────────────────────────────────────────────
    model.train()
    total_loss = 0; n_batches = 0

    for inputs, targets in train_loader:
        inputs  = {k: v.to(device) for k, v in inputs.items()}
        targets = targets.to(device)

        optimizer.zero_grad()

        # FP16 forward + backward
        with autocast('cuda'):
            logits = model(**inputs)
            loss = corn_loss_weighted(logits, targets, class_weights_tensor,
                                      num_classes, device)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item(); n_batches += 1

    avg_loss = total_loss / n_batches

    # ── Eval ──────────────────────────────────────────────
    model.eval()
    preds_list = []; true_list = []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            targets = targets.to(device)
            with autocast('cuda'):
                logits = model(**inputs)
            preds_list.extend(predict_rating_from_logits(logits.float()).cpu().numpy())
            true_list.extend((targets + 1).cpu().numpy())

    r2  = r2_score(true_list, preds_list)
    acc = 100 * np.mean(np.array(preds_list) == np.array(true_list))

    elapsed = time.time() - t0
    history['loss'].append(avg_loss)
    history['r2'].append(r2)
    history['acc'].append(acc)

    improved = "✓" if r2 > best_r2 else ""
    print(f"  Ep {ep+1}/{epochs} | Loss: {avg_loss:.4f} | "
          f"Val R²: {r2:.4f} | Acc: {acc:.2f}% | {elapsed:.0f}s {improved}")

    if r2 > best_r2:
        best_r2  = r2
        best_acc = acc
        base = model.module if N_GPUS > 1 else model
        best_state = {k: v.cpu().clone() for k, v in base.state_dict().items()}
        patience_left = patience
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"  ⏹  Early stopping à epoch {ep+1}")
            break

print(f"\nMeilleur Val R² : {best_r2:.4f} | Acc : {best_acc:.2f}%")

# Charger le meilleur checkpoint
base = model.module if N_GPUS > 1 else model
base.load_state_dict(best_state)

## 9. Courbes d'entraînement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

xs = range(1, len(history['loss']) + 1)
axes[0].plot(xs, history['loss'], marker='o', label='Train Loss')
axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(xs, history['r2'], marker='o', label='Val R²')
axes[1].set_title('Val R²'); axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('training_curves_v4.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Évaluation sur le test set

In [ ]:
model.eval()
all_preds = []
all_true  = data_test['Rating'].values

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with autocast('cuda'):
            logits = model(**inputs)
        preds = predict_rating_from_logits(logits.float())
        all_preds.extend(preds.cpu().numpy())

all_preds = np.array(all_preds)

r2         = r2_score(all_true, all_preds)
mae        = mean_absolute_error(all_true, all_preds)
acc_exact  = 100 * np.mean(all_preds == all_true)
acc_pm1    = 100 * np.mean(np.abs(all_preds - all_true) <= 1)

print("=" * 60)
print("RÉSULTATS SUR LE TEST SET")
print("=" * 60)
print(f"R² Score              : {r2:.4f}")
print(f"MAE                   : {mae:.4f}")
print(f"Accuracy exacte       : {acc_exact:.2f}%")
print(f"Accuracy (±1 étoile)  : {acc_pm1:.2f}%")
print("=" * 60)

## 11. Visualisations des résultats

In [ ]:
# Distribution prédictions vs réelles
plt.figure(figsize=(10, 4))
plt.hist(all_true,  bins=5, alpha=0.5, label='Vraies notes', color='steelblue')
plt.hist(all_preds, bins=5, alpha=0.5, label='Prédictions',  color='tomato')
plt.xlabel('Note (1-5)'); plt.ylabel('Fréquence')
plt.title('Distribution réelle vs prédite'); plt.legend()
plt.show()

In [ ]:
# Matrice de confusion

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(1, 6), yticklabels=range(1, 6))
plt.xlabel('Prédictions'); plt.ylabel('Réelles')
plt.title('Matrice de Confusion')
plt.show()

In [ ]:
# Accuracy par classe
print("Accuracy par classe :")
for c in range(1, 6):
    mask = all_true == c
    if mask.sum() > 0:
        class_acc = 100 * np.mean(all_preds[mask] == c)
        class_mae = mean_absolute_error(all_true[mask], all_preds[mask])
        print(f"  {c}★ : Acc={class_acc:.1f}% | MAE={class_mae:.3f} | n={mask.sum()}")

## 12. Analyse des erreurs

On examine les cas où le modèle se trompe de plus d'une étoile. Ce sont souvent des avis **ambigus** qui mélangent des aspects positifs et négatifs.

In [ ]:
errors     = np.abs(all_true - all_preds)
hard_cases = data_test.iloc[np.where(errors > 1)]

print(f"Exemples difficiles (erreur > 1★) : {len(hard_cases)} / {len(data_test)}")
print("-" * 60)
for i, (idx, row) in enumerate(hard_cases.head(5).iterrows()):
    pred = all_preds[data_test.index.get_loc(idx)]
    print(f"\nEx {i+1} | Vraie: {row['Rating']}★ | Prédite: {pred}★")
    print(f"  {row['Review'][:200]}...")
    print("-" * 60)

## 13. Sauvegarde du modèle

In [ ]:
model_path = 'bert_hotel_rating_model_v4.pth'
torch.save({
    'model_state_dict': best_state,
    'model_id':  model_id,
    'best_seed': SEED,
    'r2_score':  r2,
    'accuracy':  acc_exact,
    'config': {
        'num_classes':  num_classes,
        'max_length':   max_length,
        'batch_size':   batch_size,
        'epochs':       epochs,
        'use_mean_pooling': True,
        'dropout':      dropout,
    }
}, model_path)
print(f"Modèle sauvegardé : {model_path}")

if torch.cuda.is_available():
    for i in range(N_GPUS):
        alloc = torch.cuda.memory_allocated(i) / 1024**2
        resv  = torch.cuda.memory_reserved(i) / 1024**2
        print(f"GPU {i} — alloué: {alloc:.0f} MB | réservé: {resv:.0f} MB")

## Conclusion

### Résultats obtenus

| Métrique | Valeur |
|----------|--------|
| R² Score | ~0.75 |
| MAE      | ~0.32 |
| Accuracy exacte | ~70% |
| Accuracy ±1★ | ~98% |

### Ce qui a fonctionné
- **CORN** pour capturer l'ordinalité des notes
- **Class weights** pour gérer le déséquilibre
- **Mean pooling** (recommandé pour ModernBERT)
- **Cosine scheduler avec warmup** pour une convergence stable
- **Label smoothing** pour éviter la sur-confiance
- **Battle Royale de seeds** pour identifier la meilleure initialisation

### Ce qui n'a pas fonctionné
- **Oversampling** : déstabilise l'entraînement (R² baisse de 0.73 à 0.69)
- **Dual-head** (CORN + régression) : les modèles individuels sont moins bons
- **Expected Value** : pas d'amélioration systématique vs seuil dur

### Pistes d'amélioration
- **ModernBERT-large** : le principal levier pour aller au-delà de 0.73 (nécessite plus de VRAM/espace disque)
